# Evidencias individuales de Trayectoria

Construye `data/evidencias/trayectoria/evidencias_trayectoria.csv` con el esquema común
`evidencia_id | persona_id | tipo_id | texto | atributos`. Las fechas (`fecha_inicio`, `fecha_fin`) y el detalle de
cada periodo (`periodos`) van dentro de `atributos`.

El mapeo de columnas y las decisiones están en `README.md` (esta carpeta). Esta etapa no genera embeddings.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "evidencias").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evidencias.trayectoria import construir_evidencias_trayectoria

pd.set_option("display.max_colwidth", 200)

In [2]:
evidencias, reporte, validacion = construir_evidencias_trayectoria(guardar=True)
print(json.dumps(reporte, ensure_ascii=False, indent=2))

{
  "fecha_corte": "2026-09-28",
  "poblacion_base": 3195,
  "personas_excluidas_de_todo_analisis": 24,
  "poblacion_considerada": 3171,
  "personas_sin_trayectoria": 7,
  "personas_con_evidencias": 3164,
  "excluidas_inicio_futuro": 0,
  "excluidas_identicas": 0,
  "evidencias_total": 18351,
  "personas_no_consideradas_por_motivo": {
    "identificada en defunción": 24,
    "registros descartados por reglas de calidad (p.ej. solo contratos de un dia)": 5,
    "sin registros de trayectoria en ninguna fuente": 2
  },
  "subtipos": {
    "TRAYECTORIA_CARGO_ESTRUCTURAL": {
      "fuente": "trayectorias/tramos_rol.csv",
      "filas_fuente_poblacion": 9567,
      "periodos_consolidados": 6545,
      "periodos_de_un_solo_dia_excluidos": 50,
      "periodos_inicio_futuro_excluidos": 0,
      "periodos_en_evidencias": 6495,
      "columna_NIVELDOCENCIA_TRAMO_disponible": false,
      "evidencias": 5384,
      "personas": 2454,
      "evidencias_finales": 5384
    },
    "TRAYECTORIA_CONTRATO_

## Validación del esquema (todas las filas deben tener `n_fallas = 0`)

In [3]:
validacion

,chequeo,n_fallas,ok
0,columnas_esperadas,0,True
1,evidencia_id_nulo,0,True
2,evidencia_id_duplicado,0,True
3,persona_fuera_de_poblacion,0,True
4,tipo_id_invalido,0,True
5,atributos_json_invalido,0,True
6,fecha_inicio_nula,0,True
7,fechas_invertidas_sin_marcar,0,True
8,fecha_inicio_futura,0,True
9,texto_vacio,0,True


## Resumen por subtipo

In [4]:
attrs = evidencias["atributos"].map(json.loads)
resumen = evidencias.assign(
    sin_fecha_fin=attrs.map(lambda a: a.get("fecha_fin") is None),
    n_periodos=attrs.map(lambda a: a.get("n_periodos", 1)),
).groupby("tipo_id").agg(
    evidencias=("evidencia_id", "size"),
    personas=("persona_id", "nunique"),
    periodos=("n_periodos", "sum"),
    evidencias_con_varios_periodos=("n_periodos", lambda s: int((s > 1).sum())),
    sin_fecha_fin=("sin_fecha_fin", "sum"),
    textos_distintos=("texto", "nunique"),
)
resumen

,evidencias,personas,periodos,evidencias_con_varios_periodos,sin_fecha_fin,textos_distintos
tipo_id,,,,,,
TRAYECTORIA_CARGO_ESTRUCTURAL,5384,2454,6495,586,1255,1409
TRAYECTORIA_CONTRATO_PUNTUAL,3759,1932,7339,1161,243,276
TRAYECTORIA_EXPERIENCIA_EXTERNA,7994,2481,7994,0,520,7556
TRAYECTORIA_FUNCION_ADICIONAL,1214,601,2583,469,151,475


## Ejemplos por subtipo (texto + atributos)

In [5]:
for tipo_id, grupo in evidencias.groupby("tipo_id"):
    print("\n=====", tipo_id)
    for _, r in grupo.sample(3, random_state=7).iterrows():
        print(f"- {r.evidencia_id} | {r.texto}")
        print("  ", json.dumps(json.loads(r.atributos), ensure_ascii=False))


===== TRAYECTORIA_CARGO_ESTRUCTURAL
- TRY-EST-9e757ee6069a | PROFESOR NO TITULAR OCASIONAL 1 (TP), FACULTAD DE CIENCIAS NATURALES Y MATEMÁTICAS (FCNM)
   {"cargo": "PROFESOR NO TITULAR OCASIONAL 1 (TP)", "unidad": "FACULTAD DE CIENCIAS NATURALES Y MATEMÁTICAS", "unidad_sigla": "FCNM", "categorias_cargo": ["DOCENTE_NO_TITULAR_OCASIONAL"], "tipos_empleado": ["DOCENTE"], "niveles_docencia": [], "n_contratos": 5, "fecha_inicio": "2014-10-01", "fecha_fin": "2017-03-17", "vigente": false, "n_periodos": 1, "duracion_total_anios": 2.46, "periodos": [{"inicio": "2014-10-01", "fin": "2017-03-17", "duracion_anios": 2.46, "es_paralelo": false, "es_significativo": true, "nivel_docencia": null}], "_origen": {"fuente": "trayectorias/tramos_rol.csv (consolidado cargo+unidad, receso <=90 dias)", "n_tramos_rol": 1}}
- TRY-EST-e280365ceb94 | TÉCNICO DE INVESTIGACIÓN 3 (TC), CENTRO DE TECNOLOGÍAS DE LA INFORMACIÓN (CTI)
   {"cargo": "TÉCNICO DE INVESTIGACIÓN 3 (TC)", "unidad": "CENTRO DE TECNOLOGÍAS DE L

## Atributos expandidos (útil para revisar coberturas por subtipo)

In [6]:
def atributos_expandidos(tipo_id: str) -> pd.DataFrame:
    sub = evidencias[evidencias["tipo_id"] == tipo_id]
    attrs = pd.json_normalize(sub["atributos"].map(json.loads).tolist())
    return pd.concat([sub[["evidencia_id", "persona_id"]].reset_index(drop=True), attrs], axis=1)

for tipo_id in sorted(evidencias["tipo_id"].unique()):
    df = atributos_expandidos(tipo_id)
    print(f"\n{tipo_id}: % nulos por atributo")
    print((df.isna().mean() * 100).round(1).to_string())


TRAYECTORIA_CARGO_ESTRUCTURAL: % nulos por atributo


evidencia_id             0.0
persona_id               0.0
cargo                    0.0
unidad                  32.5
unidad_sigla            32.5
categorias_cargo         0.0
tipos_empleado           0.0
niveles_docencia         0.0
n_contratos              0.0
fecha_inicio             0.0
fecha_fin               23.3
vigente                  0.0
n_periodos               0.0
duracion_total_anios     0.0
periodos                 0.0
_origen.fuente           0.0
_origen.n_tramos_rol     0.0

TRAYECTORIA_CONTRATO_PUNTUAL: % nulos por atributo
evidencia_id             0.0
persona_id               0.0
contrato                 0.0
contrato_generico        0.0
unidad                  34.0
unidad_sigla            34.0
categorias               0.0
tipos_empleado           0.0
niveles_docencia         0.0
n_contratos              0.0
fecha_inicio             0.0
fecha_fin                6.5
vigente                  0.0
n_periodos               0.0
duracion_total_anios     0.0
periodos           


TRAYECTORIA_EXPERIENCIA_EXTERNA: % nulos por atributo
evidencia_id                               0.0
persona_id                                 0.0
fecha_inicio                               0.0
fecha_fin                                  6.5
cargo                                      0.0
institucion                                0.4
pais                                       0.0
es_exterior                                0.0
tipo_institucion                           0.0
relacion_laboral                           5.4
tiempo_dedicacion                          0.0
categoria_experiencia                      0.0
rol_academico                             93.6
duracion_anios                             6.5
institucion_es_espol                       0.0
_origen.fuente                             0.0
_origen.idhistorialaboral                  0.0
_origen.ids_historialaboral_fusionados     0.0
_origen.ref_evidencia                     25.6
_origen.ref_evaluaciones                  99.7

TRAY

## Textos más repetidos (baja información semántica)

In [7]:
evidencias["texto"].value_counts().head(15)

texto
Contrato civil de profesor (posgrado)                                                                                683
PROFESOR PREGRADO                                                                                                    250
Contrato de servicios profesionales                                                                                  209
Contrato de servicios profesionales, DIRECCIÓN DE ADMISIONES (ADM)                                                   195
Contrato de servicios profesionales (ejecución de actividades de investigación), DECANATO DE INVESTIGACIÓN (DINV)    122
TÉCNICO DOCENTE 3 (TC), DIRECCIÓN DE ADMISIONES (ADM)                                                                119
PROFESOR NO TITULAR OCASIONAL (TC), FACULTAD DE CIENCIAS NATURALES Y MATEMÁTICAS (FCNM)                              109
Contrato de servicios profesionales, DECANATO DE GRADO (DG)                                                          102
Contrato de servicios prof

## Una persona completa (cambiar el ID para revisar otros casos)

In [8]:
PERSONA = evidencias["persona_id"].iloc[0]
una = evidencias[evidencias["persona_id"] == PERSONA]
attrs = una["atributos"].map(json.loads)
una.assign(
    fecha_inicio=attrs.map(lambda a: a.get("fecha_inicio")),
    fecha_fin=attrs.map(lambda a: a.get("fecha_fin")),
    n_periodos=attrs.map(lambda a: a.get("n_periodos", 1)),
)[["tipo_id", "fecha_inicio", "fecha_fin", "n_periodos", "texto"]]

,tipo_id,fecha_inicio,fecha_fin,n_periodos,texto
0,TRAYECTORIA_CARGO_ESTRUCTURAL,1999-03-03,2004-09-25,2,PROFESOR
1,TRAYECTORIA_EXPERIENCIA_EXTERNA,2007-09-03,2010-12-31,1,"BECARIO, UNIVERSIDAD DE NAVARRA, España"
2,TRAYECTORIA_CARGO_ESTRUCTURAL,2014-08-01,2026-08-07,10,"PROFESOR TITULAR AGREGADO 3 (TC), FACULTAD DE INGENIERÍA MECÁNICA Y CIENCIAS DE LA PRODUCCIÓN (FIMCP)"
3,TRAYECTORIA_CONTRATO_PUNTUAL,2015-01-01,2015-10-23,2,"Contrato de servicios profesionales, FACULTAD DE INGENIERÍA MECÁNICA Y CIENCIAS DE LA PRODUCCIÓN (FIMCP)"
4,TRAYECTORIA_FUNCION_ADICIONAL,2016-01-04,2022-08-31,2,"Miembro del consejo directivo, FACULTAD DE INGENIERÍA MECÁNICA Y CIENCIAS DE LA PRODUCCIÓN (FIMCP)"
5,TRAYECTORIA_CONTRATO_PUNTUAL,2016-04-22,2016-06-11,1,"Contrato de servicios profesionales (actividades de capacitación), FACULTAD DE INGENIERÍA MECÁNICA Y CIENCIAS DE LA PRODUCCIÓN (FIMCP)"
6,TRAYECTORIA_FUNCION_ADICIONAL,2016-12-09,2017-10-27,2,"Coordinador de carrera (Ingeniería Mecánica; Mecánica), FACULTAD DE INGENIERÍA MECÁNICA Y CIENCIAS DE LA PRODUCCIÓN (FIMCP)"
7,TRAYECTORIA_CONTRATO_PUNTUAL,2020-03-06,NaN,7,Contrato civil de profesor (posgrado)
8,TRAYECTORIA_CARGO_ESTRUCTURAL,2021-03-01,2021-07-17,1,SUBDECANO(A)
9,TRAYECTORIA_CARGO_ESTRUCTURAL,2021-08-23,NaN,4,"SUBDECANO(A), FACULTAD DE INGENIERÍA MECÁNICA Y CIENCIAS DE LA PRODUCCIÓN (FIMCP)"
